# 🏙️ StreetPLM — Eixample Walkability Analyser
### Google Colab Notebook · free-tier T4 GPU optimised

**What this notebook does:**
1. Queries the **Overture Maps** pedestrian walk edges from **BigQuery** for the
   **2 km × 2 km** Eixample study area (Passeig de Gràcia centre,
   same filter as `Backend/Environment/overture_to_duckdb.py`)
2. Samples points every **250 m** along each walk edge, heading aligned with street direction
3. Checks Street View availability and fetches one **640×640** image per sample point
4. Runs **Meta PerceptionLM-1B** (full-image urban analysis) on each image
5. Saves one **JSON + JPG** per location to Google Drive —
   results join directly with Overture Maps `walk_edges` via geocoordinates

**Before running:**
- Runtime → Change runtime type → **GPU (T4)**
- Add three Colab Secrets (🔑 left sidebar → *Secrets*):  
  `GOOGLE_STREETVIEW_API_KEY` — Google Maps Platform Street View Static API  
  `HF_TOKEN` — Hugging Face token with approved access to `facebook/Perception-LM-1B`  
  `GCP_PROJECT_ID` — your GCP project with BigQuery API enabled (for querying Overture Maps)

In [ ]:
# @title ① Install dependencies  (run once per session, ~60 s)
!pip install -q \
    google-cloud-bigquery db-dtypes \
    "transformers>=4.40" \
    huggingface_hub \
    pydantic \
    Pillow requests pyproj shapely geopandas tqdm numpy

print("✅ Dependencies installed")

In [ ]:
# @title ② Mount Google Drive  (results persist across sessions)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
OUTPUT_ROOT = "/content/drive/MyDrive/UrbanABM_StreetPLM"
IMAGES_DIR  = f"{OUTPUT_ROOT}/images"
RESULTS_DIR = f"{OUTPUT_ROOT}/results"

for d in (IMAGES_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

print(f"📁 Output root : {OUTPUT_ROOT}")
print(f"   images/     : {IMAGES_DIR}")
print(f"   results/    : {RESULTS_DIR}")

Mounted at /content/drive
📁 Output root : /content/drive/MyDrive/UrbanABM_StreetPLM
   images/     : /content/drive/MyDrive/UrbanABM_StreetPLM/images
   results/    : /content/drive/MyDrive/UrbanABM_StreetPLM/results


In [ ]:
# @title ③ Configuration
from google.colab import userdata

# ── Bounding box — 2 km × 2 km centred on Passeig de Gràcia ─────────────
# Mirrors Backend/Environment/overture_to_duckdb.py
BBOX = {
    "min_lon": 2.1500,
    "min_lat": 41.3862,
    "max_lon": 2.1740,
    "max_lat": 41.4042,
}

# ── API keys — stored in Colab Secrets, never in code ────────────────────
STREETVIEW_API_KEY = userdata.get("GOOGLE_STREETVIEW_API_KEY")
HF_TOKEN           = userdata.get("HF_TOKEN")
GCP_PROJECT_ID     = userdata.get("GCP_PROJECT_ID")

# ── BigQuery / Overture Maps ────────────────────────────────────────────
BIGQUERY_PROJECT   = "bigquery-public-data"
OVERTURE_DATASET   = "overture_maps"

# ── Pipeline parameters ───────────────────────────────────────────────
SAMPLE_DISTANCE_M = 250       # metres between sample points along walk edges
SV_SIZE           = "640x640"
SV_FOV            = 90
SV_PITCH          = 0
SV_RADIUS         = 50        # panorama search radius in metres
MODEL_ID          = "facebook/Perception-LM-1B"

# ── Free-tier GPU memory budget (T4 = 15 GB) ─────────────────────
# MAX_NUM_TILES=4 keeps peak VRAM <= 6 GB so generation has plenty of headroom.
# Increase to 9 once confirmed stable (9 tiles -> 2560 image tokens).
MAX_NUM_TILES  = 4
MAX_NEW_TOKENS = 1536   # full scene JSON is ~800-1200 tokens

assert STREETVIEW_API_KEY, "Set GOOGLE_STREETVIEW_API_KEY in Colab Secrets"
assert HF_TOKEN,           "Set HF_TOKEN in Colab Secrets"
assert GCP_PROJECT_ID,     "Set GCP_PROJECT_ID in Colab Secrets"

print("✅ Configuration ready")
print(f"   BBOX          : {BBOX}")
print(f"   Sample step   : {SAMPLE_DISTANCE_M} m")
print(f"   GCP project   : {GCP_PROJECT_ID}")
print(f"   Model         : {MODEL_ID}")
print(f"   MAX_NUM_TILES : {MAX_NUM_TILES}")

✅ Configuration ready
   BBOX          : {'min_lon': 2.15, 'min_lat': 41.3862, 'max_lon': 2.174, 'max_lat': 41.4042}
   Sample step   : 100 m
   Model         : facebook/Perception-LM-1B
   MAX_NUM_TILES : 4


In [ ]:
# @title ④ Query Overture walk edges (BigQuery) & sample points at 250 m
import json
import numpy as np
import pyproj
from google.colab import auth
from google.cloud import bigquery
from shapely import wkt as shapely_wkt
from shapely.geometry import LineString
import geopandas as gpd

auth.authenticate_user()

UTM31N = "EPSG:32631"   # metric CRS for Barcelona (UTM Zone 31N)

POINTS_FILE = f"{OUTPUT_ROOT}/sample_points.json"

if os.path.exists(POINTS_FILE):
    with open(POINTS_FILE) as f:
        sample_points = json.load(f)
    print(f"♻️  Re-using {len(sample_points)} saved sample points from Drive")
else:
    # ── Query Overture Maps walk edges from BigQuery ───────────────────
    print("Querying Overture Maps walk edges from BigQuery …")
    bq_client = bigquery.Client(project=GCP_PROJECT_ID)

    bq_query = f"""
    SELECT
        id,
        ST_AsText(geometry) AS wkt,
        names.primary       AS name,
        subtype             AS road_type,
        class               AS road_class
    FROM `{BIGQUERY_PROJECT}.{OVERTURE_DATASET}.segment`
    WHERE (subtype = 'pedestrian'
           OR class IN ('pedestrian', 'footway', 'path', 'steps'))
      AND bbox.xmin >= {BBOX["min_lon"]}
      AND bbox.ymin >= {BBOX["min_lat"]}
      AND bbox.xmax <= {BBOX["max_lon"]}
      AND bbox.ymax <= {BBOX["max_lat"]}
    """

    df = bq_client.query(bq_query).to_dataframe()
    print(f"   {len(df)} walk edges returned from BigQuery")

    # ── Parse WKT into Shapely geometries & build GeoDataFrame ────────
    df["geometry"] = df["wkt"].apply(shapely_wkt.loads)
    gdf_edges = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

    # ── Reproject to metric CRS for interpolation ─────────────────────
    gdf_proj    = gdf_edges.to_crs(UTM31N)
    transformer = pyproj.Transformer.from_crs(UTM31N, "EPSG:4326", always_xy=True)

    sample_points = []
    seen_cells    = set()   # 4-decimal deduplication grid (~11 m)

    for _, row in gdf_proj.iterrows():
        geom   = row.geometry
        length = geom.length
        n_steps = max(1, int(length / SAMPLE_DISTANCE_M))

        for i in range(n_steps + 1):
            dist   = min(i * SAMPLE_DISTANCE_M, length)
            p_proj = geom.interpolate(dist)

            # Road bearing at this point -> Street View heading
            offset = 1.0 if dist + 1.0 < length else -1.0
            p2     = geom.interpolate(dist + offset)
            dx, dy = p2.x - p_proj.x, p2.y - p_proj.y
            heading = (np.degrees(np.arctan2(dx, dy)) + 360) % 360

            lon, lat = transformer.transform(p_proj.x, p_proj.y)

            cell_key = (round(lat, 4), round(lon, 4))
            if cell_key in seen_cells:
                continue
            seen_cells.add(cell_key)

            name = row["name"] if isinstance(row["name"], str) else ""
            rt   = row["road_type"] if isinstance(row["road_type"], str) else ""

            sample_points.append({
                "id"                : f"{lat:.6f}_{lon:.6f}",
                "lat"               : round(lat, 6),
                "lon"               : round(lon, 6),
                "heading"           : round(heading, 1),
                "edge_id"           : str(row["id"]),
                "dist_along_edge_m" : round(dist, 1),
                "street_name"       : name,
                "highway_type"      : rt,
            })

    with open(POINTS_FILE, 'w') as f:
        json.dump(sample_points, f, indent=2)
    print(f"✅ {len(sample_points)} unique sample points saved to Drive")

print(f"\nTotal points to process : {len(sample_points)}")
print("Preview (first 3):")
for p in sample_points[:3]:
    print(f"  lat={p['lat']}  lon={p['lon']}  "
          f"heading={p['heading']:.0f}°  street={p['street_name']}")

In [ ]:
# @title ⑤ Street View fetch helpers
import requests
import time
from pathlib import Path

_SV_BASE   = "https://maps.googleapis.com/maps/api/streetview"
_META_BASE = "https://maps.googleapis.com/maps/api/streetview/metadata"


def sv_available(lat: float, lon: float) -> bool:
    """Check panorama availability — metadata calls are free (no billing)."""
    r = requests.get(_META_BASE, params={
        "location": f"{lat},{lon}",
        "radius"  : SV_RADIUS,
        "key"     : STREETVIEW_API_KEY,
    }, timeout=10)
    return r.status_code == 200 and r.json().get("status") == "OK"


def fetch_sv(lat: float, lon: float, heading: float):
    """
    Download one Street View image to IMAGES_DIR.
    Returns local Path on success, None if unavailable or error.
    Skips download if the file already exists (idempotent).
    """
    fname = f"sv_{lat:.6f}_{lon:.6f}_h{int(heading)}.jpg"
    fpath = Path(IMAGES_DIR) / fname

    if fpath.exists():
        return fpath

    if not sv_available(lat, lon):
        return None

    try:
        r = requests.get(_SV_BASE, params={
            "size"    : SV_SIZE,
            "location": f"{lat},{lon}",
            "heading" : heading,
            "pitch"   : SV_PITCH,
            "fov"     : SV_FOV,
            "key"     : STREETVIEW_API_KEY,
        }, timeout=30)
        r.raise_for_status()
        # Grey placeholder from Google is < 5 KB
        if len(r.content) < 5_120:
            return None
        fpath.write_bytes(r.content)
        return fpath
    except Exception as exc:
        print(f"  ⚠️  SV fetch error ({lat},{lon}): {exc}")
        return None


print("✅ Street View helpers ready")

✅ Street View helpers ready


In [ ]:
# @title ⑥ Image preprocessing
# Full-image analysis — no quadrant cropping needed.
# AutoProcessor (loaded in cell ⑦) handles image tiling and normalisation.

from PIL import Image as _PILImage

print("✅ PIL ready — full-image mode (no quadrant cropping)")

In [ ]:
# @title ⑦ Load PerceptionLM-1B  (3-5 min on first run; cached after)
import gc
import os
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from huggingface_hub import login

login(token=HF_TOKEN, add_to_git_credential=False)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if DEVICE == "cuda" else torch.float32

if DEVICE == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU  : {torch.cuda.get_device_name(0)}  ({vram_gb:.1f} GB)")
    if vram_gb >= 16:
        MAX_NUM_TILES = min(MAX_NUM_TILES, 9)
    elif vram_gb >= 10:
        MAX_NUM_TILES = min(MAX_NUM_TILES, 4)
    else:
        MAX_NUM_TILES = min(MAX_NUM_TILES, 2)
    os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
else:
    print("⚠️  No GPU — inference will be very slow on CPU")

print(f"Tile budget  : {MAX_NUM_TILES}")
print(f"Loading {MODEL_ID} …")

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
processor.image_processor.max_num_tiles = MAX_NUM_TILES

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    token             = HF_TOKEN,
    torch_dtype       = DTYPE,
    device_map        = DEVICE,
    low_cpu_mem_usage = True,
)
# PLM omits lm_head.weight — manually bridge to embed_tokens
model.lm_head.weight = model.model.language_model.embed_tokens.weight
model.eval()

if DEVICE == "cuda":
    torch.cuda.empty_cache()
    gc.collect()

print("✅ PerceptionLM-1B loaded and ready")

In [ ]:
# @title ⑧ PLM prompt + inference — full-image scene analysis
import re
import json as _json
import time
from pydantic import BaseModel, field_validator

# ── Pydantic schema ──────────────────────────────────────────────────────

class StreetSceneAnalysis(BaseModel):
    """Validated schema for a full street-view image analysis."""

    # ─ Urban elements (with spatial location embedded in text) ─
    scene_overview       : str = "unknown"
    buildings            : str = "unknown"
    materials            : str = "unknown"
    building_condition   : str = "unknown"
    street_furniture     : str = "unknown"
    vegetation           : str = "unknown"
    signage              : str = "unknown"
    ground_surfaces      : str = "unknown"

    # ─ Spatial characteristics ─
    spatial_enclosure    : str = "unknown"
    pedestrian_activity  : str = "unknown"
    lighting_atmosphere  : str = "unknown"

    # ─ Viewer-type subjective perspectives ─
    as_resident          : str = "unknown"
    as_commuter          : str = "unknown"
    as_tourist           : str = "unknown"
    as_student           : str = "unknown"

    @field_validator("*", mode="before")
    @classmethod
    def _coerce(cls, v):
        if isinstance(v, (list, dict)):
            if isinstance(v, list):
                return ", ".join(str(x).strip() for x in v if str(x).strip()) or "unknown"
            return str(v)
        s = str(v).strip() if v else "unknown"
        return s if s else "unknown"


# ── Key normalisation ────────────────────────────────────────────────────

_KEY_ALIASES = {
    # scene overview
    "scene_overview": "scene_overview",
    "scene overview": "scene_overview",
    "overview": "scene_overview",
    "scene": "scene_overview",
    "description": "scene_overview",
    "scene_description": "scene_overview",
    # buildings
    "buildings": "buildings",
    "building": "buildings",
    "building_typology": "buildings",
    "building typology": "buildings",
    "building_description": "buildings",
    # materials
    "materials": "materials",
    "material": "materials",
    # building condition
    "building_condition": "building_condition",
    "building condition": "building_condition",
    "condition": "building_condition",
    # street furniture
    "street_furniture": "street_furniture",
    "street furniture": "street_furniture",
    "streetfurniture": "street_furniture",
    "furniture": "street_furniture",
    # vegetation
    "vegetation": "vegetation",
    "greenery": "vegetation",
    "trees": "vegetation",
    # signage
    "signage": "signage",
    "signs": "signage",
    # ground surfaces
    "ground_surfaces": "ground_surfaces",
    "ground surfaces": "ground_surfaces",
    "surfaces": "ground_surfaces",
    "pavement": "ground_surfaces",
    "ground": "ground_surfaces",
    # spatial enclosure
    "spatial_enclosure": "spatial_enclosure",
    "spatial enclosure": "spatial_enclosure",
    "enclosure": "spatial_enclosure",
    "spatial_impression": "spatial_enclosure",
    "spatial impression": "spatial_enclosure",
    # pedestrian activity
    "pedestrian_activity": "pedestrian_activity",
    "pedestrian activity": "pedestrian_activity",
    "activity": "pedestrian_activity",
    "pedestrians": "pedestrian_activity",
    # lighting / atmosphere
    "lighting_atmosphere": "lighting_atmosphere",
    "lighting atmosphere": "lighting_atmosphere",
    "lighting": "lighting_atmosphere",
    "atmosphere": "lighting_atmosphere",
    "light": "lighting_atmosphere",
    # viewer perspectives
    "as_resident": "as_resident",
    "as resident": "as_resident",
    "resident": "as_resident",
    "resident_perspective": "as_resident",
    "as_commuter": "as_commuter",
    "as commuter": "as_commuter",
    "commuter": "as_commuter",
    "commuter_perspective": "as_commuter",
    "as_tourist": "as_tourist",
    "as tourist": "as_tourist",
    "tourist": "as_tourist",
    "tourist_perspective": "as_tourist",
    "as_student": "as_student",
    "as student": "as_student",
    "student": "as_student",
    "student_perspective": "as_student",
}


def _normalise_result(raw: dict) -> dict:
    """Re-key a raw PLM dict and validate with Pydantic."""
    flat = {}
    for k, v in raw.items():
        if isinstance(v, dict):
            flat.update(v)
        else:
            flat[k] = v

    normalised = {}
    for k, v in flat.items():
        canonical = _KEY_ALIASES.get(k.strip().lower())
        if canonical:
            normalised[canonical] = v

    try:
        return StreetSceneAnalysis(**normalised).model_dump()
    except Exception:
        return StreetSceneAnalysis().model_dump()


# ── PLM prompt ───────────────────────────────────────────────────────────

_SCENE_PROMPT = (
    "You are an urban design expert analysing a street-view photograph "
    "from Barcelona's Eixample district.\n"
    "\n"
    "Study the image carefully. Write FULL DESCRIPTIVE SENTENCES for every field, "
    "always stating WHERE each element appears (left, right, center, foreground, "
    "background, upper, lower). Never use bare keyword lists.\n"
    "\n"
    "BAD example — too vague:\n"
    '  "materials": "concrete, brick, glass"\n'
    "GOOD example — specific and located:\n"
    '  "materials": "Left building: cream rendered facade with stone quoins at the '
    "corners. Right: exposed red brick upper floors above a polished granite shopfront. "
    'Foreground: hexagonal ceramic pavement tiles."\n'
    "\n"
    "Return ONLY a JSON object with exactly these 15 keys. "
    "Every value must be 1-3 full sentences describing THIS specific photo:\n"
    "{\n"
    '  "scene_overview": "Describe the overall scene — street type, width, '
    'dominant features, and general atmosphere.",\n'
    '  "buildings": "Describe EACH building visible — style, height, use. '
    'State whether it is on the left, right, center, or background.",\n'
    '  "materials": "Name specific facade materials and textures you can see, '
    'stating which building each belongs to.",\n'
    '  "building_condition": "Describe the maintenance state of each visible '
    'facade — restoration, weathering, damage, graffiti, etc.",\n'
    '  "street_furniture": "List benches, streetlights, bollards, bins, bike '
    'racks, etc. and state exactly where each one sits in the frame.",\n'
    '  "vegetation": "Describe trees, hedges, planters, and green areas. Note '
    'species if recognisable and their position in the image.",\n'
    '  "signage": "Describe shop signs, traffic signs, plaques, awnings — '
    'note their position and what they say if legible.",\n'
    '  "ground_surfaces": "Describe pavement materials, road surface, '
    'crosswalks, kerb types, and their positions.",\n'
    '  "spatial_enclosure": "How enclosed or open does the street feel? '
    'Mention building heights relative to street width and sky visibility.",\n'
    '  "pedestrian_activity": "Describe any people visible — what they are '
    'doing, where they are standing or walking, and the overall density.",\n'
    '  "lighting_atmosphere": "Describe time of day, shadow direction, light '
    'quality, and overall mood of the scene.",\n'
    '  "as_resident": "What would a local resident notice or value about '
    'this street? Mention specific amenities or comfort features visible.",\n'
    '  "as_commuter": "How walkable is this stretch? Comment on pavement '
    'width, obstacles, sightlines, and route efficiency.",\n'
    '  "as_tourist": "What would catch a visitor' + "'" + 's eye? Mention '
    'architectural highlights, photo-worthy details, or cultural cues.",\n'
    '  "as_student": "Comment on social spaces, seating availability, '
    'nearby shops or cafés, and overall affordability cues visible."\n'
    "}\n"
    "\n"
    "IMPORTANT: Write 1-3 complete sentences per field using specific details "
    "from THIS photograph. Do not use keyword lists. "
    "Mention spatial positions (left, right, foreground, background) in every field.\n"
    "No markdown, no explanation — output ONLY the JSON object."
)


# ── JSON parsing ─────────────────────────────────────────────────────────

def _parse_scene_json(text: str) -> dict:
    """Parse the scene analysis JSON from raw model output."""
    candidate = None

    # Try direct parse, then with closing brace
    for attempt in (text, text + "}"):
        try:
            candidate = _json.loads(attempt)
            break
        except _json.JSONDecodeError:
            pass

    # Extract JSON from surrounding text
    if candidate is None:
        m = re.search(r"\{[\s\S]*\}", text)
        if m:
            try:
                candidate = _json.loads(m.group())
            except _json.JSONDecodeError:
                pass

    # Repair truncated JSON
    if candidate is None and "{" in text:
        truncated = text.strip().rstrip(",")
        opens = truncated.count("{") - truncated.count("}")
        if opens > 0:
            try:
                candidate = _json.loads(truncated + "}" * opens)
            except _json.JSONDecodeError:
                pass
        if candidate is None:
            stripped = re.sub(r',\s*"[^"]*$', '', truncated)
            opens2 = stripped.count("{") - stripped.count("}")
            if opens2 >= 0:
                try:
                    candidate = _json.loads(stripped + "}" * opens2)
                except _json.JSONDecodeError:
                    pass

    # Fallback: key-value extraction
    if candidate is None:
        pairs = {}
        for m in re.finditer(r'"([^"]+)"\s*:\s*"([^"]*)"', text):
            pairs[m.group(1)] = m.group(2)
        if pairs:
            candidate = pairs

    if candidate and isinstance(candidate, dict):
        return _normalise_result(candidate)
    return StreetSceneAnalysis().model_dump()


# ── PLM inference — single full-image pass ───────────────────────────────

def _infer_scene(image):
    """Run PLM on the full street-view image. Returns (parsed_dict, raw_text)."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": _SCENE_PROMPT},
            ],
        }
    ]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    text += "{"
    inputs = processor(images=image, text=text, return_tensors="pt")
    # Pass ALL processor outputs — image_sizes, aspect_ratio_*, etc. are
    # required for PerceptionLM to attend to image tiles correctly.
    print(f"  Processor output keys: {list(inputs.keys())}")
    for k in list(inputs.keys()):
        v = inputs[k]
        if hasattr(v, "to"):
            if v.is_floating_point():
                inputs[k] = v.to(DEVICE, dtype=DTYPE)
            else:
                inputs[k] = v.to(DEVICE)
    with torch.no_grad():
        gen_ids = model.generate(
            **inputs,
            max_new_tokens     = MAX_NEW_TOKENS,
            eos_token_id       = processor.tokenizer.eos_token_id,
            pad_token_id       = processor.tokenizer.pad_token_id,
            repetition_penalty = 1.1,
        )

    new_ids  = gen_ids[:, inputs["input_ids"].shape[1]:]
    raw_text = "{" + processor.tokenizer.batch_decode(new_ids, skip_special_tokens=True)[0]

    if DEVICE == "cuda":
        del gen_ids, new_ids, inputs
        torch.cuda.empty_cache()
        import gc; gc.collect()

    return _parse_scene_json(raw_text), raw_text


# ── Main entry point ────────────────────────────────────────────────────

def analyze_image(image_path):
    """
    Run PerceptionLM-1B on a full street-view image.
    Returns a dict with scene analysis, viewer perspectives, and _latency_ms.
    """
    img = _PILImage.open(image_path).convert("RGB")

    t0 = time.perf_counter()
    result, raw = _infer_scene(img)
    latency_ms = (time.perf_counter() - t0) * 1000

    # Count populated fields
    populated = sum(
        1 for k, v in result.items()
        if v not in ("unknown", "", None) and not k.startswith("_")
    )
    total = sum(1 for k in result if not k.startswith("_"))
    print(f"  Scene analysis: {int(latency_ms)}ms  {populated}/{total} fields populated")
    print(f"  Raw output ({len(raw)} chars): {raw[:150]!r}")

    result["_latency_ms"] = round(latency_ms, 1)
    return result


print("✅ Inference helpers ready (full-image scene analysis mode)")

In [ ]:
# @title 🧪 Trial run — analyse ONE location (run before the full pipeline)
# Adjust TRIAL_INDEX to pick a different point, or supply manual coords.
# Cells ①–④ are NOT required — set TRIAL_MANUAL_LAT/LON to run standalone.
# This cell does NOT write anything to Google Drive — it is purely diagnostic.
TRIAL_INDEX      = 0        # index into sample_points (0 = first point)
TRIAL_MANUAL_LAT = None     # float to override, e.g. 41.3952
TRIAL_MANUAL_LON = None     # float to override, e.g. 2.1620
TRIAL_HEADING    = None     # float to override computed heading (0–360)

from IPython.display import display, Image as IPImage
from datetime import datetime, timezone
import json as _jt

# ---- resolve target point -----------------------------------------------
_UNKNOWN = {
    'street_name'      : 'unknown',
    'highway_type'     : 'unknown',
    'edge_id'          : 'unknown',
    'dist_along_edge_m': None,
}

if TRIAL_MANUAL_LAT is not None and TRIAL_MANUAL_LON is not None:
    _pt = {
        'id'     : f'{TRIAL_MANUAL_LAT:.6f}_{TRIAL_MANUAL_LON:.6f}',
        'lat'    : TRIAL_MANUAL_LAT,
        'lon'    : TRIAL_MANUAL_LON,
        'heading': TRIAL_HEADING if TRIAL_HEADING is not None else 0.0,
        **_UNKNOWN,
    }
else:
    try:
        _sp = sample_points
    except NameError:
        _sp = []
    if not _sp:
        print('⚠️  No sample_points loaded and no manual coordinates set.')
        print('Either run cells ①–④ first, or set TRIAL_MANUAL_LAT and TRIAL_MANUAL_LON above.')
        raise SystemExit('Set manual coordinates to continue.')
    _pt = dict(_sp[TRIAL_INDEX % len(_sp)])
    if TRIAL_HEADING is not None:
        _pt['heading'] = TRIAL_HEADING

print(f"Trial point [{TRIAL_INDEX}]")
print(f"  Street  : {_pt['street_name']}  ({_pt['highway_type']})")
print(f"  Lat/Lon : {_pt['lat']:.6f}, {_pt['lon']:.6f}")
print(f"  Heading : {_pt['heading']:.1f}°")
print()

# ---- fetch Street View image --------------------------------------------
print('Fetching Street View image...')
_img = fetch_sv(_pt['lat'], _pt['lon'], _pt['heading'])

if _img is None:
    print('⚠️  No Street View coverage at this location.')
    print('Try a different TRIAL_INDEX or set TRIAL_MANUAL_LAT/LON.')
else:
    print(f'Image path: {_img}')
    display(IPImage(filename=str(_img)))

    # ---- PLM inference (full-image scene analysis) ----------------------
    print('\nRunning PLM scene analysis...')
    _t0 = time.time()
    _res = analyze_image(_img)
    _lat_ms = round((time.time() - _t0) * 1000)

    # ---- display scene analysis ----------------------------------------
    _elem_keys = [
        'scene_overview', 'buildings', 'materials', 'building_condition',
        'street_furniture', 'vegetation', 'signage', 'ground_surfaces',
        'spatial_enclosure', 'pedestrian_activity', 'lighting_atmosphere',
    ]
    _viewer_keys = ['as_resident', 'as_commuter', 'as_tourist', 'as_student']
    _viewer_labels = {
        'as_resident': '🏠 Resident',
        'as_commuter': '🚶 Commuter',
        'as_tourist' : '📸 Tourist',
        'as_student' : '🎓 Student',
    }

    print(f'\n{"═" * 70}')
    print(f'  🏙️  STREET SCENE ANALYSIS')
    print(f'{"═" * 70}')

    for _k in _elem_keys:
        _v = _res.get(_k, 'unknown')
        if _v != 'unknown':
            _label = _k.replace('_', ' ').title()
            print(f'\n  {_label}:')
            # Word-wrap at ~65 chars
            _words = _v.split()
            _line = '    '
            for _w in _words:
                if len(_line) + len(_w) + 1 > 68:
                    print(_line)
                    _line = '    ' + _w
                else:
                    _line += ' ' + _w if _line.strip() else '    ' + _w
            if _line.strip():
                print(_line)

    print(f'\n{"═" * 70}')
    print(f'  👁️  VIEWER PERSPECTIVES')
    print(f'{"═" * 70}')

    for _k in _viewer_keys:
        _v = _res.get(_k, 'unknown')
        _label = _viewer_labels[_k]
        print(f'\n  {_label}:')
        if _v != 'unknown':
            _words = _v.split()
            _line = '    '
            for _w in _words:
                if len(_line) + len(_w) + 1 > 68:
                    print(_line)
                    _line = '    ' + _w
                else:
                    _line += ' ' + _w if _line.strip() else '    ' + _w
            if _line.strip():
                print(_line)
        else:
            print('    (no perspective generated)')

    print(f'\n{"═" * 70}')

    # ---- quality summary ------------------------------------------------
    _filled = sum(
        1 for _k, _v in _res.items()
        if _v not in ('unknown', '', None) and not _k.startswith('_')
    )
    _total = sum(1 for _k in _res if not _k.startswith('_'))
    print(f'\nFields populated: {_filled}/{_total}')
    if _filled < _total // 2:
        print('⚠️  Low field coverage — PLM may have struggled with this image.')

    _record = {
        'metadata': {
            'timestamp'         : datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S'),
            'latitude'          : _pt['lat'],
            'longitude'         : _pt['lon'],
            'heading'           : _pt['heading'],
            'street_name'       : _pt['street_name'],
            'highway_type'      : _pt['highway_type'],
            'edge_id'           : _pt['edge_id'],
            'dist_along_edge_m' : _pt['dist_along_edge_m'],
            'source_image'      : _img.name,
            'model'             : MODEL_ID,
            'device'            : DEVICE,
            'latency_ms'        : _res.pop('_latency_ms', _lat_ms),
            'status'            : 'trial_ok',
        },
        'scene_analysis': _res,
    }

    print(f'\n✅ PLM complete in {_lat_ms} ms')
    print('\n── Result JSON ───────────────────────────────────────────────────────────────')
    print(_jt.dumps(_record, indent=2, ensure_ascii=False))
    print('\n⚠️  Results are NOT saved to Drive. Run cell ⑨ to start the full pipeline.')

In [ ]:
# @title ⑨ Main pipeline — fetch images ➜ PLM analysis ➜ save JSON
# Checkpointing: already-saved result files are skipped automatically on re-run.
from datetime import datetime, timezone
from pathlib import Path
from tqdm.notebook import tqdm

def _result_path(point_id: str) -> Path:
    return Path(RESULTS_DIR) / f"{point_id}_analysis.json"

# ── determine remaining work ──────────────────────────────────────────────
pending = [pt for pt in sample_points if not _result_path(pt["id"]).exists()]
done    = len(sample_points) - len(pending)

print(f"Total sample points : {len(sample_points)}")
print(f"Already completed   : {done}")
print(f"Remaining           : {len(pending)}")

if not pending:
    print("\n✅ All points already analysed — nothing to do.")
else:
    stats = {"fetched": 0, "no_sv": 0, "analysed": 0, "errors": 0}

    for pt in tqdm(pending, desc="Analysing", unit="loc"):
        try:
            img_path = fetch_sv(pt["lat"], pt["lon"], pt["heading"])

            if img_path is None:
                stats["no_sv"] += 1
                stub = {
                    "metadata": {
                        "timestamp"         : datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S"),
                        "latitude"          : pt["lat"],
                        "longitude"         : pt["lon"],
                        "heading"           : pt["heading"],
                        "street_name"       : pt["street_name"],
                        "highway_type"      : pt["highway_type"],
                        "edge_id"           : pt["edge_id"],
                        "dist_along_edge_m" : pt["dist_along_edge_m"],
                        "source_image"      : None,
                        "model"             : MODEL_ID,
                        "device"            : DEVICE,
                        "status"            : "no_streetview",
                    },
                    "scene_analysis": None,
                }
                _result_path(pt["id"]).write_text(
                    _json.dumps(stub, indent=2, ensure_ascii=False), encoding="utf-8"
                )
                continue

            stats["fetched"] += 1
            time.sleep(0.25)

            scene_result = analyze_image(img_path)
            stats["analysed"] += 1

            record = {
                "metadata": {
                    "timestamp"         : datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S"),
                    "latitude"          : pt["lat"],
                    "longitude"         : pt["lon"],
                    "heading"           : pt["heading"],
                    "street_name"       : pt["street_name"],
                    "highway_type"      : pt["highway_type"],
                    "edge_id"           : pt["edge_id"],
                    "dist_along_edge_m" : pt["dist_along_edge_m"],
                    "source_image"      : img_path.name,
                    "model"             : MODEL_ID,
                    "device"            : DEVICE,
                    "latency_ms"        : scene_result.pop("_latency_ms", None),
                    "status"            : "ok",
                },
                "scene_analysis": scene_result,
            }

            _result_path(pt["id"]).write_text(
                _json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8"
            )

        except Exception as exc:
            stats["errors"] += 1
            print(f"\n  ❌ Error at {pt['id']}: {exc}")

    print("\n── Run complete ─────────────────────────────────────")
    print(f"  Street View fetched : {stats['fetched']}")
    print(f"  No Street View      : {stats['no_sv']}")
    print(f"  PLM analysed        : {stats['analysed']}")
    print(f"  Errors              : {stats['errors']}")
    print(f"\n  Results → {RESULTS_DIR}")

In [ ]:
# @title ⑩ Summary — inspect saved results
from pathlib import Path

result_files = sorted(Path(RESULTS_DIR).glob("*_analysis.json"))
print(f"Total JSON files on Drive : {len(result_files)}")

ok_count = no_sv_count = err_count = 0
for rf in result_files:
    try:
        rec    = _json.loads(rf.read_text(encoding="utf-8"))
        status = rec["metadata"].get("status", "ok")
        if status == "ok":
            ok_count += 1
        elif status == "no_streetview":
            no_sv_count += 1
        else:
            err_count += 1
    except Exception:
        err_count += 1

print(f"  ✅ Analysed with PLM  : {ok_count}")
print(f"  🚫 No Street View     : {no_sv_count}")
print(f"  ❌ Parse / IO errors  : {err_count}")

ok_files = [
    f for f in result_files
    if _json.loads(f.read_text(encoding="utf-8"))["metadata"].get("status") == "ok"
]
if ok_files:
    rec  = _json.loads(ok_files[-1].read_text(encoding="utf-8"))
    meta = rec["metadata"]
    qa   = rec.get("quadrant_analysis", {})
    mc   = qa.get("middle_center", {})
    print("\nMost recent successful result:")
    print(f"  File      : {ok_files[-1].name}")
    print(f"  Location  : {meta['latitude']}, {meta['longitude']}")
    print(f"  Street    : {meta['street_name']} ({meta['highway_type']})")
    print(f"  Heading   : {meta['heading']}°")
    print(f"  Latency   : {meta['latency_ms']} ms")
    print(f"  Building  : {mc.get('building_typology','—')}")
    print(f"  Walk score: {mc.get('walkability_score','—')}/10")
    narrative = str(mc.get('narrative','—'))
    print(f"  Narrative : {narrative[:150]}")

print("\n📌 Each JSON contains geocoordinates + edge_id for direct join")
print("   with Overture Maps walk_edges table in DuckDB.")

Total JSON files on Drive : 28
  ✅ Analysed with PLM  : 28
  🚫 No Street View     : 0
  ❌ Parse / IO errors  : 0

Most recent successful result:
  File      : 41.387100_2.157200_analysis.json
  Location  : 41.3871, 2.1572
  Street    : grid_point (grid_point)
  Heading   : 0.0°
  Latency   : 45391.5 ms
  Building  : unknown
  Walk score: None/10
  Narrative : unknown

📌 Each JSON contains geocoordinates + edge_id for direct join
   with Overture Maps walk_edges table in DuckDB.
